# W01 Thu - Setup and reproducibility

**IIT414W - Artificial Intelligence Workshop - Unit I, Week 1**  
**Thursday 3 September 2026 | Technical studio: 13:55-14:40 (45 min)**

**Session outcome:** Configure a reproducible machine learning environment, implementing version control, dependency management, and fixed random seeds.

This technical studio does not require prior Formula 1 knowledge. The small timing example is synthetic and F1-inspired only: it uses fictional labels, is not race data, and is not a model or sporting evidence.

## Read the full route before running code

1. **Plan your check - 5 min.** Write one success check and one possible failure in the next cell. **How:** short notes only. **Materials:** this notebook. **Evidence:** your own stated check and risk.
2. **Health check - 10 min.** Check Python, a Jupyter kernel, Git, and package availability. **How:** run the two health-check cells in order. **Materials:** a running Jupyter notebook; do not install anything here. **Evidence:** the actual status and version messages.
3. **Seed and offline example - 10 min.** Repeat a small synthetic timing sample with `RANDOM_SEED = 414`. **How:** run the seed cell. **Materials:** Python standard library only. **Evidence:** a small table and two matching runs.
4. **Save and reproduce - 10 min.** Find the project root and create a non-sensitive evidence file. **How:** run the root and evidence cells in order. **Materials:** the course folder with `.iit414w-root`. **Evidence:** a new file under `outputs/`.
5. **Record and prepare verification - 10 min.** Record your real status, limitation, and next action. **How:** complete the written fields, then restart and run all. **Materials:** your own observed results. **Evidence:** your status and next action for peer verification and the existing exit ticket.

Times are guidance, not deadlines. Pause whenever a word or step is unclear; ask a facilitator or partner to restate the task without supplying a result. The written instructions in this notebook are the full instructions for the technical block.

## Quick glossary

- **Seed:** a starting value that makes a pseudo-random procedure repeatable when its code and inputs stay the same.
- **Kernel:** the running Python process that executes a notebook cell.
- **Dependency:** a software package that code needs.
- **Project path:** the folder location used to find and save project files.
- **Reproducibility:** another person can repeat a documented procedure and inspect the same kind of evidence.
- **F1-inspired:** a fictional timing-style example. Knowing drivers, teams, or rules is not required.

## 1. Plan your check (5 min)

**What:** identify what would count as enough technical evidence to continue today, and one possible failure.
**How:** write short notes below before you run code.
**Materials:** this notebook; no prior setup knowledge is assumed.
**Expected evidence:** your own check, risk, and first response.

- **What single check would show that your setup is reproducible enough to continue today?**
  - Running the seeded cell twice must produce byte-identical values *and* the evidence cell must write a real JSON file into `outputs/` containing observed version numbers. "The cell ran without an error" is not enough: a cell can run and still record nothing.
- **What is one failure you expect could occur on your machine?**
  - Selecting the wrong interpreter as the kernel. My system Python is 3.14.4 and has no pandas installed; only my Anaconda Python 3.13.5 has the course packages. If Jupyter attaches to the system interpreter, the package check reports pandas as missing and Friday's data cells cannot run.
- **What would you do first if that failure occurs?**
  - Read the interpreter path and version printed by the health check, switch the kernel to the Anaconda 3.13.5 environment, then re-run from the first cell rather than installing anything from inside the notebook.

**Pause point:** if the task is unclear, ask for the wording to be restated before you begin. Your notes do not block the technical checks.

## 2. Health check (10 min)

**What:** check Python 3.10+, a working Jupyter kernel, Git, and the presence or absence of course packages.  
**How:** run the next two cells in order and read each status message.  
**Materials:** this notebook open in Jupyter or JupyterLab. Do not install packages or change settings in this notebook.  
**Expected evidence:** actual version/status messages plus a suggested action for every item not passing.

**Pause point:** after the two cells run, stop and ask for clarification if any message is unfamiliar. A missing Friday package does not stop the offline example below.

In [1]:
# Health check: standard library only. This cell does not install or change anything.
import platform
import shutil
import subprocess
import sys

MINIMUM_PYTHON = (3, 10)
health_checks = []

def add_check(name, status, evidence, next_action):
    health_checks.append({
        'check': name,
        'status': status,
        'evidence': evidence,
        'next_action': next_action,
    })
    print(f'[{status}] {name}: {evidence}')
    if status != 'PASS':
        print(f'  Next action: {next_action}')

python_version = tuple(sys.version_info[:3])
python_ok = python_version >= MINIMUM_PYTHON
add_check(
    'Python 3.10+',
    'PASS' if python_ok else 'FAIL',
    f'{platform.python_version()} on {platform.system()}',
    'Install or activate Python 3.10+ and select that interpreter as the notebook kernel.',
)

try:
    active_shell = get_ipython()
except NameError:
    active_shell = None

kernel_name = type(active_shell).__name__ if active_shell is not None else 'none'
kernel_ok = kernel_name == 'ZMQInteractiveShell'
add_check(
    'Jupyter kernel',
    'PASS' if kernel_ok else 'NOT CHECKED',
    f'Active shell: {kernel_name}',
    'Open this file in Jupyter or JupyterLab, select a Python kernel, and run this cell again.',
)

git_executable = shutil.which('git')
git_version = 'NOT CHECKED'
git_ok = False
if git_executable:
    try:
        git_probe = subprocess.run(
            ['git', '--version'],
            capture_output=True,
            text=True,
            timeout=10,
            check=False,
        )
        git_version = git_probe.stdout.strip() or git_probe.stderr.strip() or 'No version text returned'
        git_ok = git_probe.returncode == 0
    except (OSError, subprocess.SubprocessError) as error:
        git_version = f'Probe failed: {type(error).__name__}'

add_check(
    'Git available',
    'PASS' if git_ok else 'FAIL',
    git_version,
    'Install Git, restart Jupyter, and run this cell again. Do not paste credentials into this notebook.',
)

print('\nA running cell shows that this kernel can execute code now. It does not prove Restart and Run All.')

[PASS] Python 3.10+: 3.13.5 on Linux
[PASS] Jupyter kernel: Active shell: ZMQInteractiveShell
[PASS] Git available: git version 2.53.0

A running cell shows that this kernel can execute code now. It does not prove Restart and Run All.


In [2]:
# Package availability: inspect metadata only. Do not import optional packages before checking them.
from importlib import metadata as importlib_metadata

course_packages = [
    ('ipykernel', 'Needed today', 'Runs the selected Python notebook kernel.'),
    ('numpy', 'Optional today', 'Useful later; the offline example below does not require it.'),
    ('pandas', 'Prepare for Friday', 'Used for tabular F1 data work.'),
    ('requests', 'Prepare for Friday', 'Useful for documented HTTP data access.'),
    ('fastf1', 'Prepare for Friday', 'F1 data ecosystem session; not used today.'),
]

def package_version(distribution_name):
    try:
        return importlib_metadata.version(distribution_name)
    except importlib_metadata.PackageNotFoundError:
        return None

package_checks = []
print('{:<12} {:<20} {}'.format('Package', 'When', 'Status / version'))
print('-' * 70)
for package_name, when_needed, purpose in course_packages:
    version = package_version(package_name)
    status = 'AVAILABLE ({})'.format(version) if version else 'MISSING'
    package_checks.append({
        'package': package_name,
        'when_needed': when_needed,
        'available': version is not None,
        'version': version or 'NOT INSTALLED',
        'purpose': purpose,
    })
    print('{:<12} {:<20} {}'.format(package_name, when_needed, status))

print('\nA missing Friday package is preparation work, not proof that the offline setup failed today.')
print('This notebook intentionally does not run pip install or make network calls.')

Package      When                 Status / version
----------------------------------------------------------------------
ipykernel    Needed today         AVAILABLE (6.29.5)
numpy        Optional today       AVAILABLE (2.1.3)
pandas       Prepare for Friday   AVAILABLE (2.2.3)
requests     Prepare for Friday   AVAILABLE (2.32.3)
fastf1       Prepare for Friday   AVAILABLE (3.8.3)

A missing Friday package is preparation work, not proof that the offline setup failed today.
This notebook intentionally does not run pip install or make network calls.


## 3. Fixed seed and offline example (10 min)

**What:** generate the same small synthetic timing table twice with `RANDOM_SEED = 414`.  
**How:** run the next cell once; it creates two samples with the same seed and parameters, then compares them.  
**Materials:** Python's standard library only. The F1-inspired labels are fictional and require no F1 knowledge.  
**Expected evidence:** a compact table of synthetic values and a PASS or FAIL comparison message.

**Pause point:** read the table labels before running the cell. Ask for a term to be restated if needed; do not guess from outside knowledge.

In [3]:
import random

RANDOM_SEED = 414

def make_synthetic_lap_sample(seed, rows=6):
    generator = random.Random(seed)
    driver_ids = ('driver_alpha', 'driver_beta', 'driver_gamma')
    return [
        {
            'driver_id': driver_ids[index % len(driver_ids)],
            'lap_number': index + 1,
            'synthetic_lap_time_s': round(80 + generator.random() * 8, 3),
        }
        for index in range(rows)
    ]

first_run = make_synthetic_lap_sample(RANDOM_SEED)
second_run = make_synthetic_lap_sample(RANDOM_SEED)
deterministic_match = first_run == second_run

print('Synthetic F1-inspired timing table (not real race data):')
print('{:<16} {:>10} {:>24}'.format('driver_id', 'lap', 'synthetic_lap_time_s'))
print('-' * 54)
for row in first_run:
    print('{:<16} {:>10} {:>24.3f}'.format(
        row['driver_id'], row['lap_number'], row['synthetic_lap_time_s']
    ))

if deterministic_match:
    print('\n[PASS] Two runs with seed {} and the same parameters match exactly.'.format(RANDOM_SEED))
else:
    print('\n[FAIL] The two runs with seed {} did not match. Do not mark this check Ready.'.format(RANDOM_SEED))

Synthetic F1-inspired timing table (not real race data):
driver_id               lap     synthetic_lap_time_s
------------------------------------------------------
driver_alpha              1                   83.379
driver_beta               2                   82.788
driver_gamma              3                   87.544
driver_alpha              4                   85.943
driver_beta               5                   81.872
driver_gamma              6                   82.154

[PASS] Two runs with seed 414 and the same parameters match exactly.


### What this check demonstrates - and what it does not

Matching outputs demonstrate that this small procedure is deterministic under the same seed and parameters in the current environment. A seed alone does **not** fix package versions, input data, code changes, operating-system differences, or undocumented settings. That is why the evidence file also records checks and versions.

**Pause point:** state one uncertainty in your own words before moving on. A matching cell is technical evidence, not proof that you understand every part of reproducibility.

## 4. Save and reproduce (10 min)

**What:** locate the verified course project path and save non-sensitive technical evidence.  
**How:** run the root cell now; after you record your status, run the evidence cell.  
**Materials:** this notebook opened from the course project folder or one of its subfolders; the `.iit414w-root` marker.  
**Expected evidence:** a new, timestamped JSON file in `outputs/` that does not overwrite another file.

**Pause point:** if the root marker is not found, do not create a new project folder. Read the displayed next action and ask for help locating the course folder.

In [4]:
# Find the course root without relying on a personal absolute path.
from pathlib import Path

def find_project_root(start_directory):
    start_directory = start_directory.resolve()
    for candidate in (start_directory, *start_directory.parents):
        if (candidate / '.iit414w-root').is_file():
            return candidate
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_DIRECTORY = None

if PROJECT_ROOT is None:
    print('[NOT CHECKED] Project root marker .iit414w-root was not found from this working directory.')
    print('Next action: open the notebook from the course project folder or a subfolder, then run this cell again.')
else:
    OUTPUT_DIRECTORY = PROJECT_ROOT / 'outputs'
    OUTPUT_DIRECTORY.mkdir(exist_ok=True)
    print('[PASS] Course root marker found.')
    print('Evidence will be saved to: outputs/<unique-file-name>.json')
    print('No personal absolute path is recorded.')

[PASS] Course root marker found.
Evidence will be saved to: outputs/<unique-file-name>.json
No personal absolute path is recorded.


## 5. Record a real result (10 min)

**What:** record one additional technical check, your actual Ready / Minor fix / Blocked status, one limitation, and the next action.
**How:** complete the written fields, then optionally type your real status and action in the next code cell before saving evidence.
**Materials:** your observed output. If you used AI during permitted work, use your own truthful [PROMPTS.md template](../../templates/PROMPTS_template_v1.md) record.
**Expected evidence:** a personal status and next action, or `NOT RECORDED` if you leave the fields blank. Blank text is not treated as evidence of learning.

- **Additional check and observed result:**
  - Beyond the supplied checks I re-opened the saved file `outputs/setup_evidence_20260904T151918_859418Z.json` and compared it against what the cells printed. The versions match (Python 3.13.5, pandas 2.2.3, numpy 2.1.3, requests 2.32.3, fastf1 3.8.3, ipykernel 6.29.5), `synthetic_sample_match` is `true` and `technical_status` is `READY`. The same file also records `restart_and_run_all` as `NOT CHECKED`, which is correct: the notebook cannot verify that for me.
- **Status:** Ready / Minor fix / Blocked
  - Ready.
- **One limitation:**
  - `READY` describes this machine on this date only. The fixed seed makes the small synthetic sample repeatable, but it does not pin package versions, operating system or input data, so it is not evidence that another person reproduces the same result elsewhere.
- **Next action and deadline:**
  - Complete the written fields in both notebooks, restart the kernel and run all cells, commit everything to a public Git repository and package the ZIP before the Lab 0 deadline, Thursday 10 September 2026 at 12:30.

**Pause point:** ask for the task to be reformulated if needed. Do not ask another person to enter a status for you.

In [5]:
# Optional: enter your real status and next action before saving evidence. Leave both blank if not recorded yet.
student_status = 'Ready'  # Allowed values: Ready, Minor fix, Blocked
student_next_action = 'Complete the written fields in both notebooks, restart the kernel and run all cells, then commit to a public repository and package the Lab 0 ZIP before 10 September 2026 12:30.'

allowed_statuses = {'ready': 'Ready', 'minor fix': 'Minor fix', 'blocked': 'Blocked'}
normalized_status = student_status.strip().lower()
if not normalized_status:
    recorded_status = 'NOT RECORDED'
elif normalized_status in allowed_statuses:
    recorded_status = allowed_statuses[normalized_status]
else:
    recorded_status = 'INVALID ENTRY - revise before relying on this record'

if recorded_status == 'NOT RECORDED':
    print('[NOT RECORDED] Enter a truthful Ready, Minor fix, or Blocked status if you want it included in the evidence file.')
elif recorded_status.startswith('INVALID'):
    print('[NOT RECORDED] {}'.format(recorded_status))
else:
    print('[RECORDED] Status: {}'.format(recorded_status))
    action_for_display = student_next_action.strip() or 'NOT RECORDED'
    print('[RECORDED] Next action: {}'.format(action_for_display))

[RECORDED] Status: Ready
[RECORDED] Next action: Complete the written fields in both notebooks, restart the kernel and run all cells, then commit to a public repository and package the Lab 0 ZIP before 10 September 2026 12:30.


In [6]:
# Save non-sensitive technical evidence. A new file is created for each execution.
from datetime import datetime, timezone
import json

required_variables = (
    'health_checks', 'package_checks', 'RANDOM_SEED', 'deterministic_match',
    'PROJECT_ROOT', 'OUTPUT_DIRECTORY', 'recorded_status', 'student_next_action',
)
missing_variables = [name for name in required_variables if name not in globals()]

if missing_variables:
    print('[NOT CHECKED] Run the technical cells above in order before saving evidence.')
    print('Missing values: {}'.format(', '.join(missing_variables)))
elif PROJECT_ROOT is None or OUTPUT_DIRECTORY is None:
    print('[NOT CHECKED] Evidence was not saved because the verified project root was not found.')
else:
    technical_ready = all(check['status'] == 'PASS' for check in health_checks) and deterministic_match
    technical_status = 'READY' if technical_ready else 'NEEDS REVIEW'
    created_utc = datetime.now(timezone.utc)
    evidence_name = 'setup_evidence_{}.json'.format(created_utc.strftime('%Y%m%dT%H%M%S_%fZ'))
    evidence_path = OUTPUT_DIRECTORY / evidence_name

    evidence = {
        'created_utc': created_utc.isoformat(),
        'random_seed': RANDOM_SEED,
        'technical_status': technical_status,
        'student_status': recorded_status,
        'student_next_action': student_next_action.strip() or 'NOT RECORDED',
        'health_checks': health_checks,
        'package_checks': package_checks,
        'synthetic_sample_match': deterministic_match,
        'restart_and_run_all': 'NOT CHECKED - complete this from a freshly restarted kernel.',
        'privacy_note': 'No tokens, usernames, email addresses, Git history, or personal paths are recorded.',
    }

    if evidence_path.exists():
        print('[NOT CHECKED] A file with this unique name already exists; no evidence was overwritten.')
    else:
        evidence_path.write_text(json.dumps(evidence, indent=2), encoding='utf-8')
        print('[SAVED] Evidence file: outputs/{}'.format(evidence_name))
        print('[RECORDED] Technical status: {}'.format(technical_status))
        print('[RECORDED] Student status: {}'.format(recorded_status))

[SAVED] Evidence file: outputs/setup_evidence_20260905T205054_256410Z.json
[RECORDED] Technical status: READY
[RECORDED] Student status: Ready


### Brief reflection

Complete this after reviewing your evidence. Keep it factual; no response is prefilled for you.

- **One decision I made myself:**
  - I ran this notebook on my Anaconda Python 3.13.5 interpreter instead of the system Python 3.14.4, because only the Anaconda environment has the course packages installed. I recorded that interpreter's versions rather than the ones listed in the supplied `requirements_w01_fri_v1.txt`, which was captured on the lecturer's Windows Python 3.12.4 machine.
- **What I verified, and what remains uncertain:**
  - Verified: the three health checks report PASS, all five packages resolve with the versions listed above, and the two seeded samples compare as equal (`deterministic_match` is `True`). Uncertain: whether these exact pins install cleanly on another operating system, and whether the loaded packages behave identically to the lecturer's older versions - Friday's lap table showed that a FastF1 version difference does change the data returned.
- **My next action:**
  - Restart the kernel, run all cells in order, and keep the newest `outputs/setup_evidence_*.json` alongside the earlier one as evidence for the Lab 0 package.

## Peer verification and close (14:40-14:50)

**What:** verify your setup from a clean kernel and prepare your own status for the existing exit ticket.  
**How:** complete the five steps below, then let a peer inspect the evidence you produced. A peer can ask questions or restate a step; they do not run the work or choose a status for you.  
**Materials:** your saved notebook, newest `outputs/setup_evidence_*.json` file, and your observed result.  
**Expected evidence:** a real Ready, Minor fix, or Blocked status plus a next action.

1. Save this notebook.
2. Use **Restart Kernel and Run All**. This is a real action, not a box the notebook can check for you.
3. Confirm that the health check, seed comparison, and evidence saving complete without an unexpected error.
4. Keep the newest evidence file. Do not overwrite another person's evidence.
5. Carry your own status and next action to the existing exit ticket.

If Jupyter or Python will not start, record the block on the exit ticket and use the planned peer support. A classmate's successful run is not evidence that your machine is ready.

### Next session

Friday's F1 data ecosystem session has its own preparation. Resolve any missing Friday packages with support before that class; do not turn this offline setup notebook into an installation script. Consult the course programme and calendar for published assessment information.